# 49 -- flip-equivariance probe, step 3: does it hold on MEMORIZED data?

**Needs a GPU pod.** Same shape as `02_flip_pair_probe.ipynb`, different population.

Step 2 measured literal answer equivariance under a horizontal flip on rung 42 ep4's
**held-out** set (never seen, in any form): **87.2%** (n=78), paired accuracy delta
**unreadable** (below `RULES §S4`'s 0.01 floor). This notebook asks the sharper version of
the same question on data the model was **actually trained on**: `train.parquet`, **92
videos**, real training supervision (rung 18's base corpus, before the promoted-video
extras rung 42 added on top). If equivariance holds up here too -- on rows the model was
trained to answer -- that is a materially stronger case that it isn't leaning on
memorisation than the held-out result alone.

**Scope, agreed before building:** not the full 2,170 transformable train rows -- a
~200-row sample, stratified by VIDEO only (every one of the 92 videos represented, at
least once), not by `(video, rule)` -- checked first: the finer stratification floors at
261 rows minimum (92 videos x up to 3 rules, every non-empty cell keeps >=1), already past
the target. Video-only stratification floors at 92 and lands close to 200. Rule-mix drift
from the pool's own proportions is real under this coarser key and is reported, not assumed
away -- see `_tools/video_subsample.py`.

**What this can and cannot show:** a train-set row that stays equivariant under a flip it
was never shown is genuine evidence of spatial generalisation surviving memorisation
pressure. A row that does NOT track the flip is ambiguous by construction -- the model may
be reciting the exact memorised answer (real prior-override) OR may simply be wrong the
same way it would be on any hard row. This notebook cannot separate those two without a
third condition (e.g. an untrained checkpoint on the same rows), so the read below is
framed as a ceiling check, not a mechanism claim.

In [1]:
# --- bootstrap -----------------------------------------------------------------
import json, logging, os, sys, time
from pathlib import Path
import pandas as pd

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "49-flip-equivariance":
    EXP = REPO / "experiments" / "49-flip-equivariance"

for p in (
    REPO / "src",
    REPO / "vendor" / "orena-focus" / "src",
    REPO / "experiments" / "24-geometric-aug" / "_models",  # flip_audit.py, owned by rung 24
    EXP / "_tools",                                          # flip_pair_runner.py, video_subsample.py
):
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(name)s %(message)s", datefmt="%H:%M:%S")

import flip_audit
from flip_pair_runner import merge_adapter, materialize_flip_pairs, answer_paired
from video_subsample import choose_by_video, report_drift
from frame.config import BaselineConfig
from frame.data import FrameProvider, load_frame_items
from frame import metrics
from frame.subsample import freeze as freeze_manifest, SubsampleConfig
from focus.data.data_models import Request, Reference, Response, save_items
from focus.evaluation.evaluator import Evaluator
from focus.evaluation.judges import TransformersJudge

print("repo:", REPO, "| exp:", EXP, "| swift on PATH:", (Path(_envbin) / "swift").exists())

repo: /workspace/repo_yyy | exp: /workspace/repo_yyy/experiments/49-flip-equivariance | swift on PATH: True


In [2]:
# --- parameters (RAW LITERALS ONLY -- papermill injects BELOW this cell) --------
SMOKE = False          # True -> ~6 rows, wiring only, no verdict
N_SMOKE = 8

RUN = "49_train_flip_v1"
TARGET_N = 200
SAMPLE_SEED = 42

# rung 42 ep4's own merged checkpoint -- deliberately the SAME path 02_flip_pair_probe.ipynb
# merges to, so if that notebook already ran on this pod, this run skips re-merging a 17 GB
# checkpoint it does not need to rebuild.
ADAPTER = ("/workspace/repo_rodri/experiments/42-merged-corpus/runs/42_merged_v1/"
           "ckpt/v0-20260814-163049/checkpoint-4848")
BASE_MODEL = "/workspace/models/qwen3-vl-8b"
DATA_ROOT = "/workspace/orena-data"
DEVICE = "cuda"
SEED = 42
N_BOOT = 4000

In [3]:
# --- derived --------------------------------------------------------------------
PAIR_RUN_DIR = EXP / "runs" / "49_flip_pair_v1"     # shared merge cache with step 2, on purpose
RUN_DIR = EXP / "runs" / RUN
MERGED_DIR = PAIR_RUN_DIR / "merged" / "checkpoint-4848"
FRAMES_DIR = RUN_DIR / "frames"
RUN_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found -- pull the QA parquets"

MANIFEST_PATH = EXP / "RESULTS_train_flip_sample_v1.csv"

In [4]:
# --- PRE-FLIGHT: the LLM judge must resolve offline, BEFORE anything expensive ------
from transformers import AutoTokenizer

_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}. Fix the env -- do NOT disable the offline "
        "flags, the whole deployment story is offline."
    ) from exc
print(f"OK judge gate: {_judge} resolves from {os.environ.get('HF_HOME')}")

OK judge gate: Qwen/Qwen3-4B resolves from /workspace/.cache/huggingface/


In [5]:
# --- 1. audit train.parquet for transformable rows -- re-derived here, not trusted from
# a prior session's number, so this run is self-contained and re-verifiable on its own ---
cfg_probe = BaselineConfig(data_root=DATA_ROOT)
train_items_all = load_frame_items(cfg_probe, splits=("train",))
print(f"train.parquet items: {len(train_items_all)}")

parts = []
for ds in ("heico", "lapchole"):
    pq = DATA_ROOT / ds / "data" / "frame" / "train.parquet"
    df = pd.read_parquet(pq)
    parts.append(flip_audit.audit_dataframe(df, split="train", dataset=ds))
train_audit = pd.concat(parts, ignore_index=True)
train_audit["qID"] = train_audit["dataset"] + "__" + train_audit["id"].astype(str)

transformable = train_audit[train_audit["disposition"] == "transformable"].copy()
assert len(transformable) == 2170, (
    f"expected 2,170 transformable train rows (rung 24's own data-card number, independently "
    f"reproduced pre-build), got {len(transformable)} -- the data changed under this notebook"
)
print(f"transformable train rows: {len(transformable)} "
      f"(rules: {transformable['rule'].value_counts().to_dict()})")

21:51:14 INFO frame.data Loaded 8000 FRAME items from heico/train
21:51:15 INFO frame.data Loaded 5748 FRAME items from lapchole/train
21:51:15 INFO frame.data Total FRAME items: 13748


train.parquet items: 13748
transformable train rows: 2170 (rules: {'fixed_quadrant_class': 1092, 'object_center_quadrant': 640, 'all_object_positions': 438})


In [6]:
# --- 2. resolve to FrameItems, sample by video, freeze the manifest -----------------
by_qid = {i.request.qID: i for i in train_items_all}
missing = set(transformable["qID"]) - set(by_qid)
assert not missing, f"{len(missing)} transformable qIDs not resolvable via load_frame_items"
transformable_items = [by_qid[q] for q in transformable["qID"]]
del train_items_all, by_qid

if MANIFEST_PATH.exists() and (Path(str(MANIFEST_PATH) + ".sha256")).exists():
    print("manifest already frozen, reusing ->", MANIFEST_PATH)
    sample_rows = pd.read_csv(MANIFEST_PATH)
else:
    sample_rows = choose_by_video(transformable_items, target_n=TARGET_N, seed=SAMPLE_SEED)
    freeze_manifest(sample_rows, SubsampleConfig(manifest_path=MANIFEST_PATH))
    print("froze manifest ->", MANIFEST_PATH)

# `transformable` (flip_audit.audit_dataframe's own output) carries NO `video` column --
# only `dataset`/`id`/capability/format/rule/question/answer fields. Video count comes from
# the FrameItems instead, which DO carry `.video_id` (bug caught live on the pod: the first
# version of this line read `transformable['video'].nunique()` and raised KeyError('video')).
n_videos_full = len({it.video_id for it in transformable_items})
print(f"sample: {len(sample_rows)} rows, {sample_rows['video'].nunique()} of "
      f"{n_videos_full} videos")
# `transformable` already carries its own `answer_format` column (flip_audit's own output) --
# report_drift needs exactly that, no re-derivation from FrameItems needed or wanted (a merge
# of two frames that BOTH have `answer_format` silently suffixes both to _x/_y instead of
# raising, which is how this looked fine until it wasn't -- caught locally, fixed here).
print("\nrule-mix drift vs the full transformable pool (video-only stratification, NOT forced):")
print(report_drift(sample_rows, transformable))

sample_qids = set(sample_rows["qID"])
if SMOKE:
    sample_qids = set(sample_rows["qID"].head(N_SMOKE))
items = [it for it in transformable_items if it.request.qID in sample_qids]
items.sort(key=lambda it: (it.dataset, it.video_id, it.frame_index))
audit_by_qid = transformable.set_index("qID")
print(f"\nthis run: {len(items)} rows")

manifest already frozen, reusing -> /workspace/repo_yyy/experiments/49-flip-equivariance/RESULTS_train_flip_sample_v1.csv
sample: 202 rows, 92 of 92 videos

rule-mix drift vs the full transformable pool (video-only stratification, NOT forced):
                 full_frac  sample_frac   drift
answer_format                                  
fo_class            0.5032       0.5396  0.0364
multiple_choice     0.2949       0.2327 -0.0623
open_ended          0.2018       0.2277  0.0259

this run: 202 rows


In [7]:
# --- merge the adapter (skips if step 2 already did it), materialize frame pairs ----
if MERGED_DIR.is_dir() and (MERGED_DIR / "config.json").exists():
    print("already merged (shared with step 2) ->", MERGED_DIR)
else:
    t0 = time.perf_counter()
    merge_adapter(BASE_MODEL, ADAPTER, MERGED_DIR)
    print(f"merged in {time.perf_counter() - t0:.0f}s ->", MERGED_DIR)

provider = FrameProvider(BaselineConfig(data_root=DATA_ROOT))
materialize_flip_pairs(items, provider, FRAMES_DIR)
print(f"frames materialized under {FRAMES_DIR} ({2 * len(items)} files)")

already merged (shared with step 2) -> /workspace/repo_yyy/experiments/49-flip-equivariance/runs/49_flip_pair_v1/merged/checkpoint-4848
frames materialized under /workspace/repo_yyy/experiments/49-flip-equivariance/runs/49_train_flip_v1/frames (404 files)


In [8]:
# --- build the paired item list, run inference: BOTH conditions, ONE session --------
pair_rows = []
for it in items:
    qid = it.request.qID
    row = audit_by_qid.loc[qid]
    pair_rows.append({"qID": qid, "condition": "orig",
                      "image_path": str(FRAMES_DIR / f"{qid}.jpg"), "question": row["question"]})
    pair_rows.append({"qID": f"{qid}__flip", "condition": "flip",
                      "image_path": str(FRAMES_DIR / f"{qid}__flip.jpg"),
                      "question": row["flipped_question"]})
pair_items = pd.DataFrame(pair_rows)
assert len(pair_items) == 2 * len(items)
print(f"{len(pair_items)} items queued")

t0 = time.perf_counter()
preds = answer_paired(MERGED_DIR, pair_items, device=DEVICE)
print(f"inference done in {(time.perf_counter() - t0) / 60:.1f} min")

n_err = int(preds["prediction"].str.startswith("Inference Error:").sum())
if n_err:
    share = n_err / len(preds)
    print(f"G-INFER: {n_err}/{len(preds)} generations failed ({share:.1%})")
    assert share <= 0.01, f"G-INFER: {share:.1%} of generations failed -- fix the engine first"
preds.to_csv(RUN_DIR / "predictions_raw.csv", index=False)
print("wrote", RUN_DIR / "predictions_raw.csv")

404 items queued


21:57:04 INFO frame.engine Loading Qwen3-VL from /workspace/repo_yyy/experiments/49-flip-equivariance/runs/49_flip_pair_v1/merged/checkpoint-4848 …
The tokenizer you are loading from '/workspace/repo_yyy/experiments/49-flip-equivariance/runs/49_flip_pair_v1/merged/checkpoint-4848' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

21:57:36 INFO frame.engine Model ready.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
21:58:07 INFO flip_pair_runner 50/404 - 0.612 s/item
21:58:35 INFO flip_pair_runner 100/404 - 0.591 s/item
21:59:05 INFO flip_pair_runner 150/404 - 0.588 s/item
21:59:33 INFO flip_pair_runner 200/404 - 0.583 s/item
22:00:01 INFO flip_pair_runner 250/404 - 0.579 s/item
22:00:38 INFO flip_pair_runner 300/404 - 0.606 s/item
22:01:16 INFO flip_pair_runner 350/404 - 0.627 s/item
22:01:54 INFO flip_pair_runner 400/404 - 0.644 s/item
22:01:58 INFO flip_pair_runner 404/404 - 0.649 s/item
22:01:58 INFO flip_pair_runner answered 404 items in 4.4 min (0.644 s/item)


inference done in 5.0 min
wrote /workspace/repo_yyy/experiments/49-flip-equivariance/runs/49_train_flip_v1/predictions_raw.csv


In [9]:
# --- score through the SAME canonical Evaluator + real judge every other rung uses ---
preds_by_qid = preds.set_index("qID")

requests, references, responses = [], [], []
for it in items:
    qid = it.request.qID
    row = audit_by_qid.loc[qid]

    requests.append(it.request)
    references.append(it.reference)
    responses.append(Response(qID=qid, content=preds_by_qid.loc[qid, "prediction"],
                              latency=float(preds_by_qid.loc[qid, "latency"])))

    fqid = f"{qid}__flip"
    requests.append(Request(qID=fqid, videoID=it.request.videoID,
                            start_time=it.request.start_time, end_time=it.request.end_time,
                            procedure_type=it.request.procedure_type,
                            question=row["flipped_question"]))
    references.append(Reference(qID=fqid, primary=it.reference.primary,
                                _format=it.reference._format, answer=row["flipped_answer"],
                                format_kwargs=it.reference.format_kwargs,
                                secondaries=it.reference.secondaries,
                                ood=it.reference.ood, clinical=it.reference.clinical))
    responses.append(Response(qID=fqid, content=preds_by_qid.loc[fqid, "prediction"],
                              latency=float(preds_by_qid.loc[fqid, "latency"])))

for ref in references:
    ref.ood = ref.qID.split("__", 1)[0] == "heico"

save_items(responses, RUN_DIR / "responses.json")
save_items(requests, RUN_DIR / "requests.json")
save_items(references, RUN_DIR / "references.json")

judge = TransformersJudge(model_name=_judge, device=DEVICE)
evaluator = Evaluator(judges=[judge], seed=SEED)
results_df, summary_df = evaluator.run(requests=requests, references=references,
                                       responses=responses, output_dir=RUN_DIR, track=None)
del judge, evaluator
print(f"scored {len(results_df)} items ({len(items)} orig + {len(items)} flip)")

22:01:58 INFO focus.evaluation.judges Loading judge model 'Qwen/Qwen3-4B' on device 'cuda'…


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

22:02:37 INFO focus.evaluation.judges Judge model 'Qwen/Qwen3-4B' ready.
22:02:37 INFO focus.evaluation.evaluator Starting evaluation: 404 requests, 404 references, 404 responses.
22:05:35 WARNING focus.evaluation.evaluator Pre-evaluation score: only 2/10 group×distribution buckets are populated; averaging over the populated buckets.
22:05:35 INFO focus.evaluation.evaluator Pre-evaluation score: 0.970 (mean over 2 group×distribution buckets).
22:05:35 INFO focus.evaluation.evaluator Evaluation complete: 391/404 correct (96.8%).
22:05:36 INFO focus.evaluation.evaluator Results saved to /workspace/repo_yyy/experiments/49-flip-equivariance/runs/49_train_flip_v1.


scored 404 items (202 orig + 202 flip)


In [10]:
# --- read 1: paired accuracy delta, video-clustered CI ------------------------------
if SMOKE:
    print("SMOKE -- wiring only, no verdict")
else:
    orig = results_df[~results_df["qID"].str.endswith("__flip")].copy()
    flip = results_df[results_df["qID"].str.endswith("__flip")].copy()
    flip["qID"] = flip["qID"].str.removesuffix("__flip")
    assert len(orig) == len(flip) == len(items)

    print(f"accuracy, original frames (train, memorised): {orig['correctness'].mean():.4f}")
    print(f"accuracy, flipped frames  (train, memorised): {flip['correctness'].mean():.4f}")

    j = orig[["qID", "video", "primary", "correctness"]].rename(columns={"correctness": "correct_a"})
    j = j.merge(flip[["qID", "correctness"]].rename(columns={"correctness": "correct_b"}), on="qID")
    j["group"] = j["primary"].map(metrics._leaf_to_group)

    rows = [{"cell": "ALL", **metrics.paired_delta_ci(j, n_boot=N_BOOT, seed=SEED)}]
    for grp in sorted(j["group"].unique()):
        rows.append({"cell": grp, **metrics.paired_delta_ci(j[j["group"] == grp],
                                                             n_boot=N_BOOT, seed=SEED)})
    ci_df = pd.DataFrame(rows)
    ci_df["excludes_zero"] = (ci_df.ci_low > 0) | (ci_df.ci_high < 0)
    print("\nflip MINUS original, video-clustered CI (train/memorised population):")
    print(ci_df.to_string(index=False))
    print(f"\nn videos in this sample: {j['video'].nunique()} (vs 8 for the held-out probe --"
          " this CI should be materially tighter if the sample behaves as designed)")

accuracy, original frames (train, memorised): 0.9950
accuracy, flipped frames  (train, memorised): 0.9406

flip MINUS original, video-clustered CI (train/memorised population):
              cell     delta    ci_low   ci_high   n  n_videos  wins_b  wins_a  excludes_zero
               ALL -0.046687 -0.087739 -0.013458 202        92       0      11           True
object_recognition -0.046687 -0.087739 -0.013458 202        92       0      11           True

n videos in this sample: 92 (vs 8 for the held-out probe -- this CI should be materially tighter if the sample behaves as designed)


In [11]:
# --- read 2: literal answer equivariance (judge-free), WITH the checks last time's ---
# overclaim skipped -- significance test and item-count breakdown computed up front.
if SMOKE:
    print("SMOKE -- wiring only, no verdict")
else:
    import re

    from scipy.stats import fisher_exact

    quadrant_rules = {"object_center_quadrant", "all_object_positions"}
    eq_rows = []
    for it in items:
        qid = it.request.qID
        rule = audit_by_qid.loc[qid, "rule"]
        if rule not in quadrant_rules:
            continue
        p_orig = str(preds_by_qid.loc[qid, "prediction"])
        p_flip = str(preds_by_qid.loc[f"{qid}__flip", "prediction"])
        expected = flip_audit.swap_left_right_quadrants(p_orig)
        gold_text = str(audit_by_qid.loc[qid, "answer"])
        n_gold_items = len(re.findall(r"\d+\.", gold_text)) if rule == "all_object_positions" else 1
        eq_rows.append({"qID": qid, "rule": rule, "orig_prediction": p_orig,
                        "flip_prediction": p_flip, "expected_if_equivariant": expected,
                        "literally_equivariant": expected.strip() == p_flip.strip(),
                        "n_gold_items": n_gold_items})
    eq_df = pd.DataFrame(eq_rows)
    rate = eq_df["literally_equivariant"].mean() if len(eq_df) else float("nan")
    print(f"literal answer equivariance, train/memorised, {len(eq_df)} rows: {rate:.1%}")
    print(eq_df.groupby("rule")["literally_equivariant"].agg(["mean", "count"]))

    tab = pd.crosstab(eq_df["rule"], eq_df["literally_equivariant"])
    if tab.shape == (2, 2):
        odds, p = fisher_exact(tab.to_numpy())
        print(f"\nFisher exact (rule x pass/fail), this population: p = {p:.4f}")
    else:
        print("\n(one rule cell empty -- Fisher's exact not computed, n too small to matter)")

    print(f"\nheld-out comparison (step 2, already scored): 87.2% (n=78) -- "
          f"compare against {rate:.1%} (n={len(eq_df)}) here, same construct, different population")

    n_multi = int((eq_df["n_gold_items"] > 1).sum())
    print(f"\nof {len(eq_df)} quadrant-answer rows, {n_multi} have >1 gold object "
          "(checked up front this time, not after the fact)")

literal answer equivariance, train/memorised, 93 rows: 87.1%
                            mean  count
rule                                   
all_object_positions    0.782609     46
object_center_quadrant  0.957447     47

Fisher exact (rule x pass/fail), this population: p = 0.0142

held-out comparison (step 2, already scored): 87.2% (n=78) -- compare against 87.1% (n=93) here, same construct, different population

of 93 quadrant-answer rows, 9 have >1 gold object (checked up front this time, not after the fact)


In [12]:
# --- persist --------------------------------------------------------------------
if not SMOKE:
    results_df.to_csv(EXP / "RESULTS_train_flip_scored.csv", index=False)
    ci_df.to_csv(EXP / "RESULTS_train_flip_paired_ci.csv", index=False)
    eq_df.to_csv(EXP / "RESULTS_train_flip_equivariance.csv", index=False)
    (EXP / "RESULTS_train_flip_summary.json").write_text(json.dumps({
        "checkpoint": "rung 42 ep4 (checkpoint-4848, submission 03)",
        "population": "train.parquet, 92 videos, video-only-stratified sample",
        "n_rows": len(items),
        "n_videos": int(j["video"].nunique()),
        "accuracy_original": float(orig["correctness"].mean()),
        "accuracy_flipped": float(flip["correctness"].mean()),
        "literal_equivariance_rate": float(rate) if rate == rate else None,
        "literal_equivariance_n": int(len(eq_df)),
        "held_out_comparison": {"literal_equivariance_rate": 0.8717948717948718, "n": 78},
    }, indent=2), encoding="utf-8")
    print("wrote RESULTS_train_flip_scored.csv, RESULTS_train_flip_paired_ci.csv, "
          "RESULTS_train_flip_equivariance.csv, RESULTS_train_flip_summary.json")
else:
    print("SMOKE -- nothing persisted")

wrote RESULTS_train_flip_scored.csv, RESULTS_train_flip_paired_ci.csv, RESULTS_train_flip_equivariance.csv, RESULTS_train_flip_summary.json


In [13]:
# --- eyeball: what did it actually SAY? (user rule: examples on every run) ----------
n_show = min(10, len(items))
for it in items[:n_show]:
    qid = it.request.qID
    row = audit_by_qid.loc[qid]
    print(f"[{row['rule']}] {qid}")
    print(f"  Q orig: {row['question']}")
    print(f"  Q flip: {row['flipped_question']}")
    print(f"  gold orig/flip: {row['answer']!r} / {row['flipped_answer']!r}")
    print(f"  pred orig/flip: {preds_by_qid.loc[qid, 'prediction']!r} / "
          f"{preds_by_qid.loc[f'{qid}__flip', 'prediction']!r}")
    print()

[all_object_positions] heico__97999
  Q orig: At timepoint 01:31:57 please provide all relative central positions of foreign objects present in the frame. Please provide the answer in the following format: “number. object type: quadrant”, where number represents an enumeration starting with 1, object type is the type of the foreign object and quadrant is one of the following options: top/left, top/right, bottom/left, bottom/right. Respond “none” in case there are no foreign objects present at timepoint 01:31:57. For example: 1. Sponge: top/left 2. Sponge: top/right 3. Needle: bottom/left
  Q flip: At timepoint 01:31:57 please provide all relative central positions of foreign objects present in the frame. Please provide the answer in the following format: “number. object type: quadrant”, where number represents an enumeration starting with 1, object type is the type of the foreign object and quadrant is one of the following options: top/left, top/right, bottom/left, bottom/right. Respon

In [14]:
res = results_df.copy()
res["qID_base"] = res["qID"].str.removesuffix("__flip")
res["condition"] = res["qID"].apply(lambda q: "flip" if q.endswith("__flip") else "orig")
res["rule"] = res["qID_base"].map(audit_by_qid["rule"])
print(res.groupby(["rule", "condition"])["correctness"].mean().unstack())

condition                   flip      orig
rule                                      
all_object_positions    0.956522  1.000000
fixed_quadrant_class    0.926606  0.990826
object_center_quadrant  0.957447  1.000000


In [15]:
eq = eq_df.copy()
eq["flip_correct"] = eq["qID"].apply(lambda q: results_df.set_index("qID").loc[f"{q}__flip", "correctness"])
print(eq.groupby(["rule", "literally_equivariant"])["flip_correct"].agg(["mean", "count"]))

                                              mean  count
rule                   literally_equivariant             
all_object_positions   False                   0.8     10
                       True                    1.0     36
object_center_quadrant False                   0.0      2
                       True                    1.0     45
